In [1]:
"""Instructor solutions to the 15-minute pair activity. Fictional teaching data."""
from pathlib import Path
import pandas as pd
DATA = Path("data")
talks = pd.read_csv(DATA / "activity_talks.csv")
speakers = pd.read_csv(DATA / "activity_speakers.csv")
speakers_conflicting = pd.read_csv(DATA / "activity_speakers_conflicting.csv")

def show(label, df):
    print("\n" + label)
    print(df.to_string(index=False))


In [2]:
# Many-to-one, from talks to speakers on speaker_id. Right keys are unique.
# A left join should retain exactly 12 rows here.


In [3]:
enriched = pd.merge(
    talks, speakers, on="speaker_id", how="left",
    validate="many_to_one", indicator=True,
)
show("Clean join", enriched)
print("Rows:", len(enriched), "Unique talk IDs:", enriched["talk_id"].is_unique)



Clean join
talk_id speaker_id  year  season  num_words speaker group    _merge
    A01         S1  2025   April       1000   Avery     A      both
    A02         S1  2025   April       1200   Avery     A      both
    A03         S1  2025 October       1400   Avery     A      both
    A04         S1  2025 October       1600   Avery     A      both
    A05         S2  2025   April       2000   Blair     B      both
    A06         S2  2025 October       2200   Blair     B      both
    A07         S2  2026   April       2400   Blair     B      both
    A08         S3  2026   April       1800   Casey     A      both
    A09         S3  2026 October       2000   Casey     A      both
    A10         S4  2026   April       1500    Drew     B      both
    A11         S4  2026 October       1700    Drew     B      both
    A12         S9  2026 October       2100     NaN   NaN left_only
Rows: 12 Unique talk IDs: True


In [4]:
unmatched = enriched.loc[enriched["_merge"].eq("left_only")]
show("Unmatched: A12 with speaker S9", unmatched)
# Speaker S5 has no talk and does not appear in this left join.



Unmatched: A12 with speaker S9
talk_id speaker_id  year  season  num_words speaker group    _merge
    A12         S9  2026 October       2100     NaN   NaN left_only


In [5]:
summary = (
    enriched.groupby("group", dropna=False, as_index=False)
    .agg(n_talks=("talk_id", "size"), mean_words=("num_words", "mean"))
)
show("Talk-weighted summary", summary.round(1))
# A: 6 talks, 1500 words. B: 5 talks, 1960 words. Unknown: 1 talk, 2100 words.
# Each talk has equal weight. A speaker with more talks contributes more often.



Talk-weighted summary
group  n_talks  mean_words
    A        6      1500.0
    B        5      1960.0
  NaN        1      2100.0


In [6]:
unexpected = pd.merge(talks, speakers_conflicting, on="speaker_id", how="left")
print("Unexpected rows:", len(unexpected))  # 16 = 8 for S1 + 8 other talks.


Unexpected rows: 16


In [7]:
duplicates = unexpected.loc[unexpected["talk_id"].duplicated(keep=False)]
show("Repeated talk rows", duplicates)
print("Affected IDs:", sorted(duplicates["talk_id"].unique()))
print("Distinct affected talks:", duplicates["talk_id"].nunique())



Repeated talk rows
talk_id speaker_id  year  season  num_words speaker group record_status
    A01         S1  2025   April       1000   Avery     A       current
    A01         S1  2025   April       1000   Avery     B       retired
    A02         S1  2025   April       1200   Avery     A       current
    A02         S1  2025   April       1200   Avery     B       retired
    A03         S1  2025 October       1400   Avery     A       current
    A03         S1  2025 October       1400   Avery     B       retired
    A04         S1  2025 October       1600   Avery     A       current
    A04         S1  2025 October       1600   Avery     B       retired
Affected IDs: ['A01', 'A02', 'A03', 'A04']
Distinct affected talks: 4


In [8]:
conflicts = speakers_conflicting.loc[
    speakers_conflicting["speaker_id"].duplicated(keep=False)
]
show("Conflicting records", conflicts)
# The supplied data dictionary defines current as authoritative for this exercise.



Conflicting records
speaker_id speaker group record_status
        S1   Avery     A       current
        S1   Avery     B       retired


In [9]:
current = speakers_conflicting.loc[speakers_conflicting["record_status"].eq("current")]
fixed = pd.merge(
    talks, current, on="speaker_id", how="left",
    validate="many_to_one", indicator=True,
)
show("Repaired join", fixed)
print("Rows:", len(fixed), "Unique talk IDs:", fixed["talk_id"].is_unique)
show("Still unmatched", fixed.loc[fixed["_merge"].eq("left_only")])
# The repair fixes duplicated matches. It supplies no missing S9 speaker record.



Repaired join
talk_id speaker_id  year  season  num_words speaker group record_status    _merge
    A01         S1  2025   April       1000   Avery     A       current      both
    A02         S1  2025   April       1200   Avery     A       current      both
    A03         S1  2025 October       1400   Avery     A       current      both
    A04         S1  2025 October       1600   Avery     A       current      both
    A05         S2  2025   April       2000   Blair     B       current      both
    A06         S2  2025 October       2200   Blair     B       current      both
    A07         S2  2026   April       2400   Blair     B       current      both
    A08         S3  2026   April       1800   Casey     A       current      both
    A09         S3  2026 October       2000   Casey     A       current      both
    A10         S4  2026   April       1500    Drew     B       current      both
    A11         S4  2026 October       1700    Drew     B       current      both
 

In [10]:
conference_metadata = pd.read_csv(DATA / "conference_metadata.csv")
with_location = pd.merge(
    talks, conference_metadata, on=["year", "season"],
    how="left", validate="many_to_one",
)
show("Composite-key result", with_location)



Composite-key result
talk_id speaker_id  year  season  num_words   location
    A01         S1  2025   April       1000 North Hall
    A02         S1  2025   April       1200 North Hall
    A03         S1  2025 October       1400 South Hall
    A04         S1  2025 October       1600 South Hall
    A05         S2  2025   April       2000 North Hall
    A06         S2  2025 October       2200 South Hall
    A07         S2  2026   April       2400  East Hall
    A08         S3  2026   April       1800  East Hall
    A09         S3  2026 October       2000  West Hall
    A10         S4  2026   April       1500  East Hall
    A11         S4  2026 October       1700  West Hall
    A12         S9  2026 October       2100  West Hall
